In [1]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info


/mnt/disk/zhangchen/miniconda3/envs/SildeReason/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [ ]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "/mnt/sdb/models/Qwen/Qwen2.5-VL/Qwen2.5-VL-72B-Instruct-AWQ",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map="auto",
)
processor = AutoProcessor.from_pretrained("/mnt/sdb/models/Qwen/Qwen2.5-VL/Qwen2.5-VL-72B-Instruct-AWQ")
# The default range for the number of visual tokens per image in the model is 4-16384.
# You can set min_pixels and max_pixels according to your needs, such as a token range of 256-1280, to balance performance and cost.
# min_pixels = 256*28*28
# max_pixels = 1280*28*28
# processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct", min_pixels=min_pixels, max_pixels=max_pixels)

In [ ]:
# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     "/mnt/sdb/models/Qwen/Qwen2.5-VL/Qwen2.5-VL-72B-Instruct",
#     torch_dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
#     device_map="auto",
# )
# processor = AutoProcessor.from_pretrained("/mnt/sdb/models/Qwen/Qwen2.5-VL/Qwen2.5-VL-72B-Instruct")

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
            },
            {"type": "text", "text": "Describe this image by Chinese."},
        ],
    }
]

In [ ]:
# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

In [ ]:
text

In [ ]:
inputs.keys()

In [ ]:
inputs['input_ids'].shape

In [ ]:
inputs['pixel_values'].shape

In [ ]:
inputs['image_grid_thw']

In [ ]:
sum(inputs['attention_mask'][0])

In [ ]:
image_inputs

In [ ]:

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

# Qwen3-14B

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model_name = "/mnt/sdb/models/Qwen/Qwen3/Qwen3-32B-FP8"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    attn_implementation="flash_attention_2",
    device_map="cuda:1"
)

In [ ]:
# prepare the model input
# prompt = "Give me a short introduction to large language model."
prompt = "请给我介绍一下AWQ 和 FB8 的大模型的区别是什么"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

In [ ]:
text

In [ ]:
model_inputs.input_ids[0].shape

In [ ]:
# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

# Qwen3-30B-A3B-Thinking-2507-FP8

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "/mnt/sdb/models/Qwen/Qwen3/Qwen3-30B-A3B-Thinking-2507-FP8"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

thinking content: Okay, the user asked for a short introduction to large language models. Let me start by recalling what I know. LLMs are a big topic, but they want it concise. So I need to cover the basics without getting too technical.

First, define what an LLM is. It's a type of AI that processes and generates human language. Mention that they're trained on massive datasets. But wait, should I specify the size? Maybe say "billions or trillions of parameters" to give a sense of scale.

Next, explain how they work. They use neural networks, specifically transformers. But the user might not know what transformers are, so maybe skip the jargon. Focus on the key point: they predict the next word based on patterns in the data. Emphasize that they don't understand language like humans do—they're pattern matchers.

Then, highlight their capabilities. Common tasks like answering questions, writing, translation. Maybe mention examples like ChatGPT or Gemini to make it relatable. But since th

In [2]:
# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content) # no opening <think> tag
print("content:", content)

thinking content: Okay, the user asked for a short introduction to large language models. Let me start by recalling what a basic LLM is. They're AI models trained on huge amounts of text data to understand and generate human-like text. But I need to keep it concise since they want it short.

Hmm, the user might be someone new to AI, maybe a student or just curious. They probably need the key points without too much jargon. Let me break it down: what they are, how they work, what they do, and why they matter.

Wait, should I mention specific examples like GPT or BERT? Maybe briefly, but since it's a short intro, maybe just reference them generally. Also, important to clarify they're not sentient—they're statistical models. Don't want to spread myths.

The user might not know terms like "neural networks" or "tokenization." So I should explain in simple terms. Like, they learn patterns from text, not understanding like humans. Emphasize the training data and the scale—billions of paramete

# GLM-4.1V-9B-Thinking

In [2]:
from transformers import AutoProcessor, Glm4vForConditionalGeneration
from PIL import Image
import requests
import torch

# Load model and processor
MODEL_PATH = "/mnt/sdb/models/GLM/GLM-4.1V/GLM-4.1V-9B-Thinking"
model = Glm4vForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path=MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_PATH, use_fast=True)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [18]:
for i in inputs.keys():
    print(i)

input_ids
attention_mask
pixel_values
image_grid_thw


In [20]:
inputs.image_grid_thw

tensor([[  1,  98, 146]], device='cuda:0')

In [4]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "url": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
            },
            {
                "type": "text",
                "text": "Describe this image by Chinese."
            }
        ],
    }
]
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)
generated_ids = model.generate(**inputs, max_new_tokens=8192)
output_text = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
print(output_text)

<think>用户现在需要用中文描述这张图片。首先看画面元素：海滩、人物、狗狗，互动场景，光线等。先分解元素：沙滩、海洋、夕阳（光线），人物穿着格子衬衫、短裤，坐在沙滩上，和浅棕色狗狗互动，狗狗戴彩色项圈，它们似乎在玩，阳光温暖，氛围轻松。然后组织语言，描述场景、人物动作、环境等。

首先，画面展现了一处海滩场景，沙滩上，一位穿着格子衬衫和短裤的人坐在沙滩上，正与一只浅棕色的狗狗互动。狗狗戴着带有彩色图案的项圈，正用前爪和人的手互动，似乎在玩耍。背景中是波光粼粼的海面，海浪轻拍沙滩，远处天空明亮，阳光洒下，给整个场景增添了温暖的氛围，整体画面充满了人与宠物之间的温馨互动，以及海滩上的惬意与自然美景，给人一种轻松、愉悦的感觉。

现在检查是否准确：人物穿着格子衬衫、短裤，狗狗的项圈，互动动作（伸手），海滩环境（沙子、海水、阳光），氛围温馨。然后调整语句流畅性。

最终描述：这张图片展现了在海滩上的温馨场景。一位穿着格子衬衫和短裤的人坐在沙滩上，正与一只浅棕色的狗狗互动。狗狗戴着带有彩色图案的项圈，正用前爪和人的手进行互动，仿佛在玩耍。背景中是波光粼粼的海面，海浪轻拍沙滩，远处天空明亮，暖黄色的阳光洒下，为整个场景增添了温暖的氛围。人与宠物之间的互动、沙滩上的自然美景，共同营造出轻松愉悦、充满爱意的氛围。</think><answer>这张图片展现了在海滩上的温馨场景。一位穿着格子衬衫和短裤的人坐在沙滩上，正与一只浅棕色的狗狗互动。狗狗戴着带有彩色图案的项圈，正用前爪和人的手进行互动，仿佛在玩耍。背景中是波光粼粼的海面，海浪轻拍沙滩，远处天空明亮，暖黄色的阳光洒下，为整个场景增添了温暖的氛围。人与宠物之间的互动、沙滩上的自然美景，共同营造出轻松愉悦、充满爱意的氛围。</answer>


# Qwen2.5-VL-72B

In [ ]:
CUDA_VISIBLE_DEVICES=0,1 vllm serve /mnt/sdb/models/Qwen/Qwen2.5-VL/Qwen2.5-VL-72B-Instruct-AWQ/ --tensor-parallel-size 2 --port 9999 --host 0.0.0.0 --gpu-memory-utilization 0.9 --max-model-len 65536 --max-seq-len-to-capture 65536 --served-model-name Qwen2.5-VL-72B-Instruct-AWQ --swap-space 16

In [ ]:
CUDA_VISIBLE_DEVICES=6,7 vllm serve /mnt/sdb/models/Qwen/Qwen2.5-VL/Qwen2.5-VL-72B-Instruct-AWQ/ --tensor-parallel-size 2 --port 9999 --host 0.0.0.0 --gpu-memory-utilization 0.5 --max-model-len 65536 --max-seq-len-to-capture 65536 --served-model-name Qwen2.5-VL-72B-Instruct-AWQ --swap-space 16

In [ ]:
CUDA_VISIBLE_DEVICES=6,7 vllm serve /mnt/data/zhangchen/models/Qwen/Qwen2.5-VL/Qwen2.5-VL-72B-Instruct-unsloth-bnb-4bit/ --tensor-parallel-size 2 --port 9999 --host 0.0.0.0 --gpu-memory-utilization 0.9 --max-model-len 65536 --max-seq-len-to-capture 65536 --served-model-name Qwen2.5-VL-72B-Instruct-unsloth-bnb-4bit --swap-space 16

In [ ]:

Qwen2.5-VL-72B-Instruct-unsloth-bnb-4bit 

In [ ]:
CUDA_VISIBLE_DEVICES=6,7 vllm serve /mnt/data/zhangchen/models/Qwen/Qwen2.5-VL/Qwen2.5-VL-72B-Instruct-GGUF/UD-Q8_K_XL/Qwen2.5-VL-72B-Instruct-UD-Q8_K_XL-00001-of-00002.gguf --tensor-parallel-size 2 --port 9999 --host 0.0.0.0 --gpu-memory-utilization 0.9 --max-model-len 65536 --max-seq-len-to-capture 65536 --served-model-name Qwen2.5-VL-72B-Instruct-GGUF0-Q8_K_XL --swap-space 16

In [ ]:
CUDA_VISIBLE_DEVICES=6,7 vllm serve /mnt/data/zhangchen/models/Qwen/Qwen2.5-VL/Qwen2.5-VL-72B-Instruct --tensor-parallel-size 2 --port 9999 --host 0.0.0.0 --gpu-memory-utilization 0.9 --max-model-len 16384 --max-seq-len-to-capture 16384 --served-model-name Qwen2.5-VL-72B-Instruct --swap-space 32

In [ ]:
/mnt/data/zhangchen/models/Qwen/Qwen2.5-VL/Qwen2.5-VL-72B-Instruct


In [ ]:
CUDA_VISIBLE_DEVICES=0,1 vllm serve /mnt/sdb/models/Qwen/Qwen2.5-VL/Qwen2.5-VL-32B-Instruct/ --tensor-parallel-size 2 --port 9999 --host 0.0.0.0 --gpu-memory-utilization 0.9 --max-model-len 32768 --served-model-name Qwen2.5-VL-32B-Instruct

In [ ]:
CUDA_VISIBLE_DEVICES=1 vllm serve /mnt/sdb/models/Qwen/Qwen2.5-VL/Qwen2.5-VL-32B-Instruct-AWQ/  --port 9999 --host 0.0.0.0 --gpu-memory-utilization 0.98 --max-model-len 32768 --served-model-name Qwen2.5-VL --quantization awq --dtype float16

# CUDA_VISIBLE_DEVICES=0 vllm serve /mnt/Hdisk/models/Qwen/Qwen3/Qwen3-32B-FP8 --enable-reasoning --reasoning-parser deepseek_r1 --port 8999 --host 0.0.0.0 --gpu-memory-utilization 0.9 --max-model-len 32768 --served-model-name Qwen3-32B-FP8
# 单卡 11 tokens /s

# Qwen3-30B-A3B-Thinking-2507

In [ ]:
CUDA_VISIBLE_DEVICES=0,1 vllm serve /mnt/sdb/models/Qwen/Qwen3/Qwen3-30B-A3B-Thinking-2507 --reasoning-parser deepseek_r1 --tensor-parallel-size 2 --port 8999 --host 0.0.0.0 --gpu-memory-utilization 0.8 --max-model-len 65536 --max-seq-len-to-capture 65536 --served-model-name Qwen3-30B-A3B-Thinking-2507 --swap-space 16

CUDA_VISIBLE_DEVICES=0 vllm serve /mnt/sdb/models/Qwen/Qwen3/Qwen3-30B-A3B-Thinking-2507-FP8 --enable-reasoning --reasoning-parser deepseek_r1 --port 8999 --host 0.0.0.0 --gpu-memory-utilization 0.9 --max-model-len 32768 --served-model-name Qwen3-30B-A3B-Thinking-2507-FP8

In [ ]:
CUDA_VISIBLE_DEVICES=0 vllm serve /mnt/sdb/models/Qwen/Qwen3/Qwen3-30B-A3B-Thinking-2507-FP8 --reasoning-parser deepseek_r1 --port 8999 --host 0.0.0.0 --gpu-memory-utilization 0.8 --max-model-len 32768 --served-model-name Qwen3-30B-A3B-Thinking-2507-FP8

In [ ]:
CUDA_VISIBLE_DEVICES=0,1 vllm serve /mnt/sdb/models/Qwen/Qwen3/Qwen3-30B-A3B-Thinking-2507 --reasoning-parser deepseek_r1 --tensor-parallel-size 2 --port 8999 --host 0.0.0.0 --gpu-memory-utilization 0.8 --max-model-len 65536 --served-model-name Qwen3-30B-A3B-Thinking-2507 -swap-space 32

# Qwen3-235B-A22B-Thinking-2507-AWQ

In [1]:
CUDA_VISIBLE_DEVICES=0,4,6,7 vllm serve /mnt/sdb/models/Qwen3-235B-A22B-Thinking-2507-AWQ --reasoning-parser deepseek_r1 --tensor-parallel-size 4 --port 8999 --host 0.0.0.0 --gpu-memory-utilization 0.5  --max-num-seqs 64 --max-model-len 65536  --max-seq-len-to-capture 65536  --served-model-name Qwen3-235B-A22B-Thinking-2507-AWQ --swap-space 16

SyntaxError: invalid decimal literal (3373575569.py, line 1)

In [ ]:
CUDA_VISIBLE_DEVICES=6,7 vllm serve /mnt/sdb/models/Qwen3-235B-A22B-Thinking-2507-AWQ --reasoning-parser deepseek_r1 --tensor-parallel-size 2 --port 8999 --host 0.0.0.0 --gpu-memory-utilization 0.95  --max-num-seqs 64 --max-model-len 65536  --max-seq-len-to-capture 65536  --served-model-name Qwen3-235B-A22B-Thinking-2507-AWQ --swap-space 16

CUDA_VISIBLE_DEVICES=0 vllm serve /mnt/sdb/models/Qwen/Qwen3/Qwen3-30B-A3B-Thinking-2507-FP8 --enable-reasoning --reasoning-parser deepseek_r1 --port 8999 --host 0.0.0.0 --gpu-memory-utilization 0.9 --max-model-len 32768 --served-model-name Qwen3-30B-A3B-Thinking-2507-FP8


# CUDA_VISIBLE_DEVICES=0 vllm serve /mnt/sdb/models/Qwen/Qwen3/Qwen3-32B-AWQ --enable-reasoning --reasoning-parser deepseek_r1 --port 8999 --host 0.0.0.0 --gpu-memory-utilization 0.9 --max-model-len 32768 --served-model-name Qwen3-32B # 单卡 40 tokens / s

# CUDA_VISIBLE_DEVICES=1 vllm serve /mnt/sdb/models/Qwen/Qwen2.5-VL/Qwen2.5-VL-32B-Instruct-AWQ/  --port 9999 --host 0.0.0.0 --gpu-memory-utilization 0.98 --max-model-len 32768 --served-model-name Qwen2.5-VL --quantization awq --dtype float16

In [ ]:
# CUDA_VISIBLE_DEVICES=0,1 vllm serve /mnt/Hdisk/models/Qwen/Qwen3/Qwen3-32B --tensor-parallel-size 2 --enable-reasoning --reasoning-parser deepseek_r1 --port 8999 --host 0.0.0.0 --gpu-memory-utilization 0.9 --max-model-len 32768 --served-model-name Qwen3-32B

# Qwen3-VL-235B-A22B-Thinking-AWQ

In [ ]:
CUDA_VISIBLE_DEVICES=0,4,6,7 vllm serve /mnt/sdb/models/Qwen3-VL-235B-A22B-Thinking-AWQ --reasoning-parser deepseek_r1 --tensor-parallel-size 4 --port 8999 --host 0.0.0.0 --gpu-memory-utilization 0.5  --max-num-seqs 64 --max-model-len 65536  --max-seq-len-to-capture 65536  --served-model-name Qwen3-VL-235B-A22B-Thinking-AWQ --swap-space 16

In [ ]:
CUDA_VISIBLE_DEVICES=6,7 vllm serve /mnt/sdb/models/Qwen3-VL-235B-A22B-Thinking-AWQ --tensor-parallel-size 2 --port 8999 --host 0.0.0.0 --gpu-memory-utilization 0.9  --max-num-seqs 1 --max-model-len 65536  --served-model-name Qwen3-VL-235B-A22B-Thinking-AWQ --swap-space 32 --enforce-eager --limit-mm-per-prompt.video 0

In [ ]:
# CUDA_VISIBLE_DEVICES=6,7 vllm serve /mnt/sdb/models/Qwen3-235B-A22B-Thinking-2507-AWQ --reasoning-parser deepseek_r1 --tensor-parallel-size 2 --port 8999 --host 0.0.0.0 --gpu-memory-utilization 0.95  --max-num-seqs 64 --max-model-len 65536  --max-seq-len-to-capture 65536  --served-model-name Qwen3-235B-A22B-Thinking-2507-AWQ --swap-space 16

# Qwen3-VL-30B-A3B-Thinking

CUDA_VISIBLE_DEVICES=0,1 vllm serve /mnt/sdb/models/Qwen/Qwen3-VL/Qwen3-VL-30B-A3B-Thinking --tensor-parallel-size 2 --port 7999 --host 0.0.0.0 --gpu-memory-utilization 0.90  --max-num-seqs 2 --max-model-len 65536  --served-model-name Qwen3-VL-30B-A3B-Thinking --swap-space 16 --limit-mm-per-prompt.video 0 --mm-encoder-tp-mode weights --enable-expert-parallel

CUDA_VISIBLE_DEVICES=0,1 vllm serve /mnt/sdb/models/Qwen/Qwen3-VL/Qwen3-VL-30B-A3B-Thinking --tensor-parallel-size 2 --port 7999 --host 0.0.0.0 --gpu-memory-utilization 0.95  --max-num-seqs 2 --max-model-len 65536  --served-model-name Qwen3-VL-30B-A3B-Thinking --swap-space 16 --limit-mm-per-prompt.video 0 --enable-expert-parallel --mm-encoder-tp-mode data

In [ ]:
CUDA_VISIBLE_DEVICES=0,1 vllm serve /mnt/sdb/models/Qwen/Qwen3-VL/Qwen3-VL-30B-A3B-Thinking --tensor-parallel-size 2 --port 7999 --host 0.0.0.0 --gpu-memory-utilization 0.95  --max-num-seqs 8 --max-model-len 65536  --served-model-name Qwen3-VL-30B-A3B-Thinking --enable-auto-tool-choice

In [ ]:
CUDA_VISIBLE_DEVICES=0,1 vllm serve /data/home/zhangchen/models/Qwen3-VL-30B-A3B-Thinking --tensor-parallel-size 2 --port 7999 --host 0.0.0.0 --gpu-memory-utilization 0.80  --max-num-seqs 20 --max-model-len 65536  --served-model-name Qwen3-VL-30B-A3B-Thinking --swap-space 16 --limit-mm-per-prompt.video 0 --mm-encoder-tp-mode data --reasoning-parser deepseek_r1 --enable-auto-tool-choice --tool-call-parser hermes --enable-expert-parallel

In [ ]:
CUDA_VISIBLE_DEVICES=0,1 \
VLLM_LOGGING_CONFIG_PATH=/tmp/vllm_prompt_details_logging.json \
vllm serve /data/home/zhangchen/models/Qwen3-VL-30B-A3B-Thinking \
  --tensor-parallel-size 2 \
  --port 7999 \
  --host 0.0.0.0 \
  --gpu-memory-utilization 0.80 \
  --max-num-seqs 20 \
  --max-model-len 65536 \
  --served-model-name Qwen3-VL-30B-A3B-Thinking \
  --swap-space 16 \
  --limit-mm-per-prompt.video 0 \
  --mm-encoder-tp-mode data \
  --reasoning-parser deepseek_r1 \
  --enable-auto-tool-choice \
  --tool-call-parser hermes \
  --enable-expert-parallel \
  --enable-log-requests \
  --max-log-len 2000 \
  --uvicorn-log-level debug

CUDA_VISIBLE_DEVICES=0,1 vllm serve /mnt/sdb/models/Qwen/Qwen3-VL/Qwen3-VL-30B-A3B-Thinking --tensor-parallel-size 2 --port 7999 --host 0.0.0.0 --gpu-memory-utilization 0.90  --max-num-seqs 2 --max-model-len 65536  --served-model-name Qwen3-VL-30B-A3B-Thinking --swap-space 16 --limit-mm-per-prompt.video 0 --mm-encoder-tp-mode weights --enable-expert-parallel  --dtype bfloat16 --kv-cache-dtype bfloat16

# Qwen3-VL-32B-Instruct

In [ ]:
CUDA_VISIBLE_DEVICES=0,1 vllm serve /data/home/zhangchen/models/Qwen3-VL-32B-Instruct --tensor-parallel-size 2 --port 7999 --host 0.0.0.0 --gpu-memory-utilization 0.80  --max-num-seqs 12 --max-model-len 65536  --served-model-name Qwen3-VL-32B-Instruct --swap-space 16 --limit-mm-per-prompt.video 0 --mm-encoder-tp-mode data --enable-auto-tool-choice --tool-call-parser hermes

In [ ]:
CUDA_VISIBLE_DEVICES=0,1 vllm serve /mnt/sdb/models/Qwen/Qwen3-VL/Qwen3-VL-32B-Instruct --tensor-parallel-size 2 --port 7999 --host 0.0.0.0 --gpu-memory-utilization 0.80  --max-num-seqs 12 --max-model-len 65536  --served-model-name Qwen3-VL-32B-Instruct --swap-space 16 --limit-mm-per-prompt.video 0 --mm-encoder-tp-mode data --enable-auto-tool-choice --tool-call-parser hermes

# Qwen3-VL-32B-Thinking

In [ ]:
CUDA_VISIBLE_DEVICES=0,1 vllm serve /data/home/zhangchen/models/Qwen3-VL-32B-Thinking --tensor-parallel-size 2 --port 7999 --host 0.0.0.0 --gpu-memory-utilization 0.80  --max-num-seqs 8 --max-model-len 65536  --served-model-name Qwen3-VL-32B-Thinking --swap-space 16 --limit-mm-per-prompt.video 0 --mm-encoder-tp-mode data --reasoning-parser deepseek_r1 --enable-auto-tool-choice --tool-call-parser hermes

In [ ]:
CUDA_VISIBLE_DEVICES=0,1 vllm serve /data/home/zhangchen/models/Qwen3-VL-32B-Thinking --tensor-parallel-size 2 --port 7999 --host 0.0.0.0 --gpu-memory-utilization 0.95  --max-num-seqs 8 --max-model-len 65536  --served-model-name Qwen3-VL-32B-Thinking --swap-space 16 --limit-mm-per-prompt.video 0 --mm-encoder-tp-mode weights --reasoning-parser deepseek_r1

In [ ]:
CUDA_VISIBLE_DEVICES=0,1 vllm serve /mnt/sdb/models/Qwen/Qwen3-VL/Qwen3-VL-32B-Thinking --tensor-parallel-size 2 --port 7999 --host 0.0.0.0 --gpu-memory-utilization 0.95  --max-num-seqs 2 --max-model-len 65536  --served-model-name Qwen3-VL-32B-Thinking

In [ ]:
CUDA_VISIBLE_DEVICES=0,1 vllm serve /mnt/sdb/models/Qwen/Qwen3-VL/Qwen3-VL-32B-Instruct --tensor-parallel-size 2 --port 7999 --host 0.0.0.0 --gpu-memory-utilization 0.95  --max-num-seqs 2 --max-model-len 65536  --served-model-name Qwen3-VL-32B-Instruct

In [ ]:
CUDA_VISIBLE_DEVICES=6,7 vllm serve /mnt/sdb/models/Qwen3-VL-32B-Thinking --tensor-parallel-size 2 --port 8999 --host 0.0.0.0 --gpu-memory-utilization 0.90  --max-num-seqs 4 --max-model-len 65536  --served-model-name Qwen3-VL-32B-Thinking --swap-space 16 --limit-mm-per-prompt.video 0 --mm-encoder-tp-mode weights --reasoning-parser deepseek_r1

In [ ]:
CUDA_VISIBLE_DEVICES=6,7 vllm serve /mnt/sdb/models/Qwen3-VL-32B-Thinking --tensor-parallel-size 2 --port 8999 --host 0.0.0.0 --gpu-memory-utilization 0.95  --max-num-seqs 16 --max-model-len 65536  --served-model-name Qwen3-VL-32B-Thinking --swap-space 16 --limit-mm-per-prompt.video 0 --mm-encoder-tp-mode data

In [ ]:
(torch29) zhangchen@user:~/project/RL$ CUDA_VISIBLE_DEVICES=0,1 vllm serve /data/home/zhangchen/models/Qwen3-VL-32B-Thinking --tensor-parallel-size 2 --port 7999 --host 0.0.0.0 --gpu-memory-utilization 0.95  --max-num-seqs 2 --max-model-len 65536  --served-model-name Qwen3-VL-32B-Thinking

# Qwen3.5

In [ ]:
CUDA_VISIBLE_DEVICES=0,1 vllm serve /mnt/sdb/models/Qwen/Qwen3.5-VL/Qwen3.5-35B-A3B \
    --tensor-parallel-size 2 --port 7999 --host 0.0.0.0 --gpu-memory-utilization 0.80  --max-num-seqs 1 --max-model-len 65536  \
    --enable-expert-parallel \
    --served-model-name Qwen3.5-35B-A3B --swap-space 32\
    --mm-encoder-tp-mode data \
    --mm-processor-cache-type shm \
    --reasoning-parser qwen3 \
    --enable-prefix-caching \
    --reasoning-parser qwen3 --enable-auto-tool-choice --tool-call-parser qwen3_coder 

In [ ]:
vllm bench serve \
  --backend openai-chat \
  --base-url http://127.0.0.1:7999 \
  --endpoint /v1/chat/completions \
  --model /data/home/zhangchen/models/Qwen3.5/Qwen3.5-122B-A10B \
  --served-model-name Qwen3.5-122B-A10B \
  --dataset-name random \
  --random-input-len 40480 \
  --random-output-len 10120 \
  --num-prompts 200 \
  --request-rate 20

In [ ]:
CUDA_VISIBLE_DEVICES=0,1,2,3 vllm serve /data/home/zhangchen/models/Qwen3.5/Qwen3.5-122B-A10B \
    --tensor-parallel-size 1 --port 7999 --host 0.0.0.0 --gpu-memory-utilization 0.90  --max-num-seqs 40 --max-model-len 65536  \
    --enable-expert-parallel --data-parallel-size 4 \
    --served-model-name Qwen3.5-122B-A10B --swap-space 0\
    --mm-encoder-tp-mode data \
    --mm-processor-cache-type shm \
    --reasoning-parser qwen3 \
    --enable-prefix-caching \
    --reasoning-parser qwen3 --enable-auto-tool-choice --tool-call-parser qwen3_coder 

In [ ]:
CUDA_VISIBLE_DEVICES=0,1,2,3 vllm serve /data/home/zhangchen/models/Qwen3.5/Qwen3.5-122B-A10B \
    --tensor-parallel-size 4 --port 7999 --host 0.0.0.0 --gpu-memory-utilization 0.90  --max-num-seqs 40 --max-model-len 65536  \
    --served-model-name Qwen3.5-122B-A10B\
    --reasoning-parser qwen3 \
    --enable-auto-tool-choice --tool-call-parser qwen3_coder 

In [ ]:
CUDA_VISIBLE_DEVICES=0,1,2,3 vllm serve /data/home/zhangchen/models/Qwen3.5/Qwen3.5-122B-A10B \
    --tensor-parallel-size 4 --port 7999 --host 0.0.0.0 --gpu-memory-utilization 0.90  --max-num-seqs 40 --max-model-len 65536  \
    --served-model-name Qwen3.5-122B-A10B --swap-space 0\
    --reasoning-parser qwen3 \
    --enable-auto-tool-choice --tool-call-parser qwen3_coder 

In [ ]:
CUDA_VISIBLE_DEVICES=0,1,2,3 vllm serve /data/home/zhangchen/models/Qwen3.5/Qwen3.5-122B-A10B \
    --tensor-parallel-size 4 --port 7999 --host 0.0.0.0 --gpu-memory-utilization 0.90  --max-num-seqs 40 --max-model-len 65536  \
    --enable-expert-parallel \
    --served-model-name Qwen3.5-122B-A10B --swap-space 0\
    --mm-encoder-tp-mode data \
    --mm-processor-cache-type shm \
    --reasoning-parser qwen3 \
    --enable-prefix-caching \
    --reasoning-parser qwen3 --enable-auto-tool-choice --tool-call-parser qwen3_coder 

# GLM4.5V-FP8

VLLM_DISABLE_TORCH_COMPILE=1 LANG="en_US.UTF-8" LC_ALL="en_US.UTF-8" PYTHONIOENCODING="utf-8" CUDA_VISIBLE_DEVICES=6,7 vllm serve    /mnt/data/zhangchen/models/GLM/GLM-4.5V-FP8     --served-model-name GLM-4.5V-FP8   --dtype bfloat16   --tool-call-parser glm45     --reasoning-parser glm45     --enable-auto-tool-choice     --allowed-local-media-path /     --enable-expert-parallel     --swap-space 32     --max-num-seqs 512     --max-model-len 32768     --max-seq-len-to-capture 32768     --gpu-memory-utilization 0.9     --tensor-parallel-size 2     --trust-remote-code     --disable-log-requests     --host 0.0.0.0     --port 9999     --media-io-kwargs '{"video": {"num_frames": -1}}'

In [ ]:
CUDA_VISIBLE_DEVICES=6,7 vllm serve  /mnt/sdb/models/GLM/GLM-4.5V-FP8 --served-model-name GLM-4.5V-FP8   --dtype bfloat16   --tool-call-parser glm45     --reasoning-parser glm45     --enable-auto-tool-choice     --allowed-local-media-path /     --enable-expert-parallel     --swap-space 32     --max-num-seqs 512     --max-model-len 65536     --max-seq-len-to-capture 65536     --gpu-memory-utilization 0.7     --tensor-parallel-size 2     --trust-remote-code     --disable-log-requests     --host 0.0.0.0     --port 9999     --media-io-kwargs '{"video": {"num_frames": -1}}'

In [ ]:
VLLM_DISABLE_TORCH_COMPILE=1 LANG="en_US.UTF-8" LC_ALL="en_US.UTF-8" PYTHONIOENCODING="utf-8" CUDA_VISIBLE_DEVICES=6,7 vllm serve    /mnt/data/zhangchen/models/GLM/GLM-4.5V-FP8     --served-model-name GLM-4.5V-FP8   --dtype bfloat16   --tool-call-parser glm45     --reasoning-parser glm45     --enable-auto-tool-choice     --allowed-local-media-path /  --swap-space 32     --max-num-seqs 512     --max-model-len 32768     --max-seq-len-to-capture 32768     --gpu-memory-utilization 0.9     --tensor-parallel-size 2     --trust-remote-code     --disable-log-requests     --host 0.0.0.0     --port 9999     --media-io-kwargs '{"video": {"num_frames": -1}}'

In [ ]:
VLLM_DISABLE_TORCH_COMPILE=1 LANG="en_US.UTF-8" LC_ALL="en_US.UTF-8" PYTHONIOENCODING="utf-8" CUDA_VISIBLE_DEVICES=6,7 vllm serve   /mnt/data/zhangchen/models/GLM/GLM-4.5V-FP8  --served-model-name GLM-4.5V-FP8   --dtype bfloat16   --tool-call-parser glm45     --reasoning-parser glm45     --enable-auto-tool-choice     --allowed-local-media-path /   --swap-space 32     --max-num-seqs 512     --max-model-len 65536     --max-seq-len-to-capture 65536     --gpu-memory-utilization 0.7     --tensor-parallel-size 2     --trust-remote-code     --disable-log-requests     --host 0.0.0.0     --port 9999     --media-io-kwargs '{"video": {"num_frames": -1}}'

In [ ]:
CUDA_VISIBLE_DEVICES=6,7 vllm serve /mnt/sdb/models/GLM/GLM-4.5V-AWQ-8bit  --served-model-name GLM-4.5V-int8   --dtype float16   --tool-call-parser glm45     --reasoning-parser glm45     --enable-auto-tool-choice     --allowed-local-media-path /     --enable-expert-parallel     --swap-space 32     --max-num-seqs 32     --max-model-len 65536     --max-seq-len-to-capture 65536     --gpu-memory-utilization 0.7     --tensor-parallel-size 2     --trust-remote-code     --disable-log-requests     --host 0.0.0.0     --port 9999     --media-io-kwargs '{"video": {"num_frames": -1}}'

In [ ]:
CUDA_VISIBLE_DEVICES=0,1 vllm serve  /mnt/sdb/models/QuantTrio/GLM/GLM-4.5V-AWQ --served-model-name GLM-4.5V-AWQ     --tool-call-parser glm45     --reasoning-parser glm45     --enable-auto-tool-choice     --allowed-local-media-path /     --enable-expert-parallel     --swap-space 32     --max-num-seqs 512     --max-model-len 65536     --max-seq-len-to-capture 65536     --gpu-memory-utilization 0.9     --tensor-parallel-size 2     --trust-remote-code     --disable-log-requests     --host 0.0.0.0     --port 9999     --media-io-kwargs '{"video": {"num_frames": -1}}'

In [ ]:
CUDA_VISIBLE_DEVICES=0,1  vllm serve  /mnt/sdb/models/cpatonn/GLM/GLM-4.5V-AWQ-4bit --dtype bfloat16 --served-model-name GLM-4.5V-AWQ-4bit   --tool-call-parser glm45     --reasoning-parser glm45     --enable-auto-tool-choice     --allowed-local-media-path /     --enable-expert-parallel     --swap-space 16     --max-num-seqs 32     --max-model-len 65536     --max-seq-len-to-capture 65536     --gpu-memory-utilization 0.9     --tensor-parallel-size 2     --trust-remote-code    --host 0.0.0.0     --port 9999     --media-io-kwargs '{"video": {"num_frames": -1}}'

# Try vllm

In [5]:
from openai import OpenAI
import openai

openai.api_key = '1111111' # 这里随便填一个
openai.base_url = 'http://10.26.65.226:8999/v1'
model_name = 'Qwen3-30B-A3B-Thinking-2507'


def get_completion(prompt, model="Qwen3-32B-FP8"):
    client = OpenAI(api_key=openai.api_key,
                    base_url=openai.base_url
                    )
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=False
    )
    reasoning_content = response.choices[0].message.reasoning_content
    content = response.choices[0].message.content
    return reasoning_content,content, response

prompt = "你是一个专业的病理医生,我现在有一个TCGA的病理报告,但是是用ocr从pdf中直接提取的, 因此可能存在问题,请帮我去除错误(例如可能存在页数被扫描进了文本或者报告的书面格式导致的问题),重新组织语言将其整理为一段话,要求是一段txt文本描述, 不能是markdown格式,报告内容如下:T. P.6/8. Anonymous No.: Gender: F. Race: White. CLINICAL HISTORY. Not provided. LMP: Hysterectomy. PRE-OP DIAGNOSIS: Right breast cancer. POST-OP DIAGNOSIS: Same. PROCEDURE: Right needle localized segmental mestectomy and right sentinel node biopsy. FINAL DIAGNOSIS. PART 1: RIGHT BREAST. 4 O' CLOCK NEEDLE LOCALIZED SEGMENTAL MASTECTOMY. A. TWO (2) FOCI OF INVASIVE DUCTALCARCINOMA NO SPECIAL TYPE. B. LARGER TUMOR NOTTINGHAM GRADE 2 (TUBULE FORMATION 3, NUCLEAR PLEOMORPHISM 2,. MITOTIC ACTIVITY 2; TOTAL SCORE: 7/9). C. SMALLER TUMOR NOTTINGHAM GRADE 2 (TUBULE FORMATION 3. NUCLEAR PLEOMORPHISM 2,. MITOTIC ACTIVITY 1; TOTAL SCORE: 6/9). THE LARGER TUMOR MEASURES 1.7 CM AND IS LOCATED AT 4 O' CLOCK. E. THE SMALLER TUMOR MEASURES 0.7 CM AND IS LOCATED AT 5 O' CLOCK. F. DUCTAL CARCINOMA IN SITU (DCIS), NUCLEAR GRADE 2. SOLID TYPE WITH MINIMAL NECROSIS. G. THE DCIS CONSTITUTES 5% OF THE TOTAL TUMOR MASS AND IS PRESENT AWAY FROM THE. INVASIVE COMPONENT. H. LYMPHOVASCULAR SPACE INVASION IS IDENTIFIED. I. INKED MARGINS ARE NEGATIVE FOR CARCINOMA. HOWEVER, INVASIVE CARCINOMA IS 1.0 MM TC. THE ANTERIOR MARGIN AND 1 5 MM TO THE POSTERIOR MARGIN (see comment). J. ATYPICAL DUCTAL HYPERPLASIA. K. FLORID DUCTAL EPITHELIAL HYPERPLASIA AND FIBROCYSTIC CHANGES WITH ASSOCIATED. MICROCALCIFICATIONS. L. PREVIOUS BIOPSY SITE CHANGES. M. ONE (1) INTRAMAMMARY IYYPH NODE, NEGATIVE FOR CARCINOMA (0/1). N. THE INVASIVE TUMOR CELLS ARE POSITIVE FOR ESTROGEN AND PROGESTERONE RECEPTORS. AND NEGATIVE FOR HER-2 (FISH NOT AMPLIFIED) AS PER PREVIOUS PATHOLOGY REPORT. PART 2: SENTINEL LYMPH NODE #1. RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA. (0/1). earcin oma, infiltrating dutal, NOS 8500/3. PART 3: SENTINEL LYMPH NODE #2, RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA (0/1). PART 4: SENTINEL LYMPH NODE #3. RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA (0/1). PART 5: RIGHT BREAST, 5 O' CLOCK, NEW MEDIAL MARGIN, EXCISION. A. NO CARCINOMA SEEN. B. BENIGN BREAST PARENCHYMA WITH FIBROCYSTIC CHANGES."



prompt = "As a professional pathologist, you will clean and reorganize the OCR-extracted TCGA pathology report into a single paragraph of plain text. The process will: Remove page numbers, headers/footers, and OCR artifacts. Correct common OCR errors. Merge fragmented text and maintain medical accuracy. Preserve all key pathological findings. Output as continuous plain text without markdown."

repot = "T. P.6/8. Anonymous No.: Gender: F. Race: White. CLINICAL HISTORY. Not provided. LMP: Hysterectomy. PRE-OP DIAGNOSIS: Right breast cancer. POST-OP DIAGNOSIS: Same. PROCEDURE: Right needle localized segmental mestectomy and right sentinel node biopsy. FINAL DIAGNOSIS. PART 1: RIGHT BREAST. 4 O' CLOCK NEEDLE LOCALIZED SEGMENTAL MASTECTOMY. A. TWO (2) FOCI OF INVASIVE DUCTALCARCINOMA NO SPECIAL TYPE. B. LARGER TUMOR NOTTINGHAM GRADE 2 (TUBULE FORMATION 3, NUCLEAR PLEOMORPHISM 2,. MITOTIC ACTIVITY 2; TOTAL SCORE: 7/9). C. SMALLER TUMOR NOTTINGHAM GRADE 2 (TUBULE FORMATION 3. NUCLEAR PLEOMORPHISM 2,. MITOTIC ACTIVITY 1; TOTAL SCORE: 6/9). THE LARGER TUMOR MEASURES 1.7 CM AND IS LOCATED AT 4 O' CLOCK. E. THE SMALLER TUMOR MEASURES 0.7 CM AND IS LOCATED AT 5 O' CLOCK. F. DUCTAL CARCINOMA IN SITU (DCIS), NUCLEAR GRADE 2. SOLID TYPE WITH MINIMAL NECROSIS. G. THE DCIS CONSTITUTES 5% OF THE TOTAL TUMOR MASS AND IS PRESENT AWAY FROM THE. INVASIVE COMPONENT. H. LYMPHOVASCULAR SPACE INVASION IS IDENTIFIED. I. INKED MARGINS ARE NEGATIVE FOR CARCINOMA. HOWEVER, INVASIVE CARCINOMA IS 1.0 MM TC. THE ANTERIOR MARGIN AND 1 5 MM TO THE POSTERIOR MARGIN (see comment). J. ATYPICAL DUCTAL HYPERPLASIA. K. FLORID DUCTAL EPITHELIAL HYPERPLASIA AND FIBROCYSTIC CHANGES WITH ASSOCIATED. MICROCALCIFICATIONS. L. PREVIOUS BIOPSY SITE CHANGES. M. ONE (1) INTRAMAMMARY IYYPH NODE, NEGATIVE FOR CARCINOMA (0/1). N. THE INVASIVE TUMOR CELLS ARE POSITIVE FOR ESTROGEN AND PROGESTERONE RECEPTORS. AND NEGATIVE FOR HER-2 (FISH NOT AMPLIFIED) AS PER PREVIOUS PATHOLOGY REPORT. PART 2: SENTINEL LYMPH NODE #1. RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA. (0/1). earcin oma, infiltrating dutal, NOS 8500/3. PART 3: SENTINEL LYMPH NODE #2, RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA (0/1). PART 4: SENTINEL LYMPH NODE #3. RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA (0/1). PART 5: RIGHT BREAST, 5 O' CLOCK, NEW MEDIAL MARGIN, EXCISION. A. NO CARCINOMA SEEN. B. BENIGN BREAST PARENCHYMA WITH FIBROCYSTIC CHANGES."
reasoning_content, content, response = get_completion(prompt + " the pathology  repot:" + repot, 
                                           model=model_name
			                                     )
print('reasoning_content',reasoning_content)


reasoning_content Hmm, the user is asking me to act as a professional pathologist to clean and reorganize an OCR-extracted TCGA pathology report. They've provided a messy text with OCR errors, page numbers, and fragmented information that needs to be transformed into a single paragraph of plain text.

Looking at this pathology report, I can see it's quite disorganized. There are page numbers like "T. P.6/8" at the beginning, headers/footers, and many OCR artifacts such as "mestectomy" instead of "mastectomy" and "TC" instead of "mm" for measurements. I need to carefully correct these while preserving all medical details.

The report describes a right breast cancer case with multiple components: two invasive ductal carcinomas (one larger at 1.7 cm, one smaller at 0.7 cm), DCIS, lymphovascular invasion, and sentinel lymph node biopsies. I notice some critical errors I must fix: "NOTTINGHAM" is misspelled as "NOTTINGHAM" in parts, "Ductal Carcinoma In Situ" is written as "DUCTAL CARCINOMA

In [3]:
from openai import OpenAI
import openai

openai.api_key = '1111111' # 这里随便填一个
openai.base_url = 'http://10.26.65.226:1234/v1'
model_name = 'qwen2.5-vl-72b-instruct'


def get_completion(prompt, model="Qwen3-32B-FP8"):
    client = OpenAI(api_key=openai.api_key,
                    base_url=openai.base_url
                    )
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=False
    )
    return response

prompt = "你是一个专业的病理医生,我现在有一个TCGA的病理报告,但是是用ocr从pdf中直接提取的, 因此可能存在问题,请帮我去除错误(例如可能存在页数被扫描进了文本或者报告的书面格式导致的问题),重新组织语言将其整理为一段话,要求是一段txt文本描述, 不能是markdown格式,报告内容如下:T. P.6/8. Anonymous No.: Gender: F. Race: White. CLINICAL HISTORY. Not provided. LMP: Hysterectomy. PRE-OP DIAGNOSIS: Right breast cancer. POST-OP DIAGNOSIS: Same. PROCEDURE: Right needle localized segmental mestectomy and right sentinel node biopsy. FINAL DIAGNOSIS. PART 1: RIGHT BREAST. 4 O' CLOCK NEEDLE LOCALIZED SEGMENTAL MASTECTOMY. A. TWO (2) FOCI OF INVASIVE DUCTALCARCINOMA NO SPECIAL TYPE. B. LARGER TUMOR NOTTINGHAM GRADE 2 (TUBULE FORMATION 3, NUCLEAR PLEOMORPHISM 2,. MITOTIC ACTIVITY 2; TOTAL SCORE: 7/9). C. SMALLER TUMOR NOTTINGHAM GRADE 2 (TUBULE FORMATION 3. NUCLEAR PLEOMORPHISM 2,. MITOTIC ACTIVITY 1; TOTAL SCORE: 6/9). THE LARGER TUMOR MEASURES 1.7 CM AND IS LOCATED AT 4 O' CLOCK. E. THE SMALLER TUMOR MEASURES 0.7 CM AND IS LOCATED AT 5 O' CLOCK. F. DUCTAL CARCINOMA IN SITU (DCIS), NUCLEAR GRADE 2. SOLID TYPE WITH MINIMAL NECROSIS. G. THE DCIS CONSTITUTES 5% OF THE TOTAL TUMOR MASS AND IS PRESENT AWAY FROM THE. INVASIVE COMPONENT. H. LYMPHOVASCULAR SPACE INVASION IS IDENTIFIED. I. INKED MARGINS ARE NEGATIVE FOR CARCINOMA. HOWEVER, INVASIVE CARCINOMA IS 1.0 MM TC. THE ANTERIOR MARGIN AND 1 5 MM TO THE POSTERIOR MARGIN (see comment). J. ATYPICAL DUCTAL HYPERPLASIA. K. FLORID DUCTAL EPITHELIAL HYPERPLASIA AND FIBROCYSTIC CHANGES WITH ASSOCIATED. MICROCALCIFICATIONS. L. PREVIOUS BIOPSY SITE CHANGES. M. ONE (1) INTRAMAMMARY IYYPH NODE, NEGATIVE FOR CARCINOMA (0/1). N. THE INVASIVE TUMOR CELLS ARE POSITIVE FOR ESTROGEN AND PROGESTERONE RECEPTORS. AND NEGATIVE FOR HER-2 (FISH NOT AMPLIFIED) AS PER PREVIOUS PATHOLOGY REPORT. PART 2: SENTINEL LYMPH NODE #1. RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA. (0/1). earcin oma, infiltrating dutal, NOS 8500/3. PART 3: SENTINEL LYMPH NODE #2, RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA (0/1). PART 4: SENTINEL LYMPH NODE #3. RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA (0/1). PART 5: RIGHT BREAST, 5 O' CLOCK, NEW MEDIAL MARGIN, EXCISION. A. NO CARCINOMA SEEN. B. BENIGN BREAST PARENCHYMA WITH FIBROCYSTIC CHANGES."



prompt = "As a professional pathologist, you will clean and reorganize the OCR-extracted TCGA pathology report into a single paragraph of plain text. The process will: Remove page numbers, headers/footers, and OCR artifacts. Correct common OCR errors. Merge fragmented text and maintain medical accuracy. Preserve all key pathological findings. Output as continuous plain text without markdown."

repot = "T. P.6/8. Anonymous No.: Gender: F. Race: White. CLINICAL HISTORY. Not provided. LMP: Hysterectomy. PRE-OP DIAGNOSIS: Right breast cancer. POST-OP DIAGNOSIS: Same. PROCEDURE: Right needle localized segmental mestectomy and right sentinel node biopsy. FINAL DIAGNOSIS. PART 1: RIGHT BREAST. 4 O' CLOCK NEEDLE LOCALIZED SEGMENTAL MASTECTOMY. A. TWO (2) FOCI OF INVASIVE DUCTALCARCINOMA NO SPECIAL TYPE. B. LARGER TUMOR NOTTINGHAM GRADE 2 (TUBULE FORMATION 3, NUCLEAR PLEOMORPHISM 2,. MITOTIC ACTIVITY 2; TOTAL SCORE: 7/9). C. SMALLER TUMOR NOTTINGHAM GRADE 2 (TUBULE FORMATION 3. NUCLEAR PLEOMORPHISM 2,. MITOTIC ACTIVITY 1; TOTAL SCORE: 6/9). THE LARGER TUMOR MEASURES 1.7 CM AND IS LOCATED AT 4 O' CLOCK. E. THE SMALLER TUMOR MEASURES 0.7 CM AND IS LOCATED AT 5 O' CLOCK. F. DUCTAL CARCINOMA IN SITU (DCIS), NUCLEAR GRADE 2. SOLID TYPE WITH MINIMAL NECROSIS. G. THE DCIS CONSTITUTES 5% OF THE TOTAL TUMOR MASS AND IS PRESENT AWAY FROM THE. INVASIVE COMPONENT. H. LYMPHOVASCULAR SPACE INVASION IS IDENTIFIED. I. INKED MARGINS ARE NEGATIVE FOR CARCINOMA. HOWEVER, INVASIVE CARCINOMA IS 1.0 MM TC. THE ANTERIOR MARGIN AND 1 5 MM TO THE POSTERIOR MARGIN (see comment). J. ATYPICAL DUCTAL HYPERPLASIA. K. FLORID DUCTAL EPITHELIAL HYPERPLASIA AND FIBROCYSTIC CHANGES WITH ASSOCIATED. MICROCALCIFICATIONS. L. PREVIOUS BIOPSY SITE CHANGES. M. ONE (1) INTRAMAMMARY IYYPH NODE, NEGATIVE FOR CARCINOMA (0/1). N. THE INVASIVE TUMOR CELLS ARE POSITIVE FOR ESTROGEN AND PROGESTERONE RECEPTORS. AND NEGATIVE FOR HER-2 (FISH NOT AMPLIFIED) AS PER PREVIOUS PATHOLOGY REPORT. PART 2: SENTINEL LYMPH NODE #1. RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA. (0/1). earcin oma, infiltrating dutal, NOS 8500/3. PART 3: SENTINEL LYMPH NODE #2, RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA (0/1). PART 4: SENTINEL LYMPH NODE #3. RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA (0/1). PART 5: RIGHT BREAST, 5 O' CLOCK, NEW MEDIAL MARGIN, EXCISION. A. NO CARCINOMA SEEN. B. BENIGN BREAST PARENCHYMA WITH FIBROCYSTIC CHANGES."
response = get_completion(prompt + " the pathology  repot:" + repot, 
                                           model=model_name
			                                     )
print('reasoning_content',response)


reasoning_content ChatCompletion(id='chatcmpl-6c763t79m94blga95yoh7q', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Anonymous No.: Gender: F. Race: White. CLINICAL HISTORY: Not provided. LMP: Hysterectomy. PRE-OP DIAGNOSIS: Right breast cancer. POST-OP DIAGNOSIS: Same. PROCEDURE: Right needle localized segmental mastectomy and right sentinel node biopsy. FINAL DIAGNOSIS: PART 1: RIGHT BREAST, 4 O' CLOCK NEEDLE LOCALIZED SEGMENTAL MASTECTOMY. Two foci of invasive ductal carcinoma no special type were identified. The larger tumor has a Nottingham grade 2 (tubule formation 3, nuclear pleomorphism 2, mitotic activity 2; total score: 7/9) and measures 1.7 cm, located at 4 o'clock. The smaller tumor also has a Nottingham grade 2 (tubule formation 3, nuclear pleomorphism 2, mitotic activity 1; total score: 6/9) and measures 0.7 cm, located at 5 o'clock. Ductal carcinoma in situ (DCIS), nuclear grade 2, solid type with minimal necrosis co

In [5]:
print('content',response.usage)

content CompletionUsage(completion_tokens=535, prompt_tokens=877, total_tokens=1412, completion_tokens_details=None, prompt_tokens_details=None)


# CUDA_VISIBLE_DEVICES=0,1 vllm serve /mnt/sdb/models/Qwen/Qwen2.5-VL/Qwen2.5-VL-72B-Instruct-AWQ/ --tensor-parallel-size 2 --port 9999 --host 0.0.0.0 --gpu-memory-utilization 0.9 --max-model-len 32768 --served-model-name Qwen2.5-VL-72B-Instruct-AWQ --quantization awq --dtype float16

In [8]:
from openai import OpenAI
import openai

openai.api_key = '1111111' # 这里随便填一个
openai.base_url = 'http://10.26.65.226:9999/v1'
model_name = 'Qwen2.5-VL-72B-Instruct-AWQ'


def get_completion(prompt, model="Qwen3-32B-FP8"):
    client = OpenAI(api_key=openai.api_key,
                    base_url=openai.base_url
                    )
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=False
    )
    return response

prompt = "你是一个专业的病理医生,我现在有一个TCGA的病理报告,但是是用ocr从pdf中直接提取的, 因此可能存在问题,请帮我去除错误(例如可能存在页数被扫描进了文本或者报告的书面格式导致的问题),重新组织语言将其整理为一段话,要求是一段txt文本描述, 不能是markdown格式,报告内容如下:T. P.6/8. Anonymous No.: Gender: F. Race: White. CLINICAL HISTORY. Not provided. LMP: Hysterectomy. PRE-OP DIAGNOSIS: Right breast cancer. POST-OP DIAGNOSIS: Same. PROCEDURE: Right needle localized segmental mestectomy and right sentinel node biopsy. FINAL DIAGNOSIS. PART 1: RIGHT BREAST. 4 O' CLOCK NEEDLE LOCALIZED SEGMENTAL MASTECTOMY. A. TWO (2) FOCI OF INVASIVE DUCTALCARCINOMA NO SPECIAL TYPE. B. LARGER TUMOR NOTTINGHAM GRADE 2 (TUBULE FORMATION 3, NUCLEAR PLEOMORPHISM 2,. MITOTIC ACTIVITY 2; TOTAL SCORE: 7/9). C. SMALLER TUMOR NOTTINGHAM GRADE 2 (TUBULE FORMATION 3. NUCLEAR PLEOMORPHISM 2,. MITOTIC ACTIVITY 1; TOTAL SCORE: 6/9). THE LARGER TUMOR MEASURES 1.7 CM AND IS LOCATED AT 4 O' CLOCK. E. THE SMALLER TUMOR MEASURES 0.7 CM AND IS LOCATED AT 5 O' CLOCK. F. DUCTAL CARCINOMA IN SITU (DCIS), NUCLEAR GRADE 2. SOLID TYPE WITH MINIMAL NECROSIS. G. THE DCIS CONSTITUTES 5% OF THE TOTAL TUMOR MASS AND IS PRESENT AWAY FROM THE. INVASIVE COMPONENT. H. LYMPHOVASCULAR SPACE INVASION IS IDENTIFIED. I. INKED MARGINS ARE NEGATIVE FOR CARCINOMA. HOWEVER, INVASIVE CARCINOMA IS 1.0 MM TC. THE ANTERIOR MARGIN AND 1 5 MM TO THE POSTERIOR MARGIN (see comment). J. ATYPICAL DUCTAL HYPERPLASIA. K. FLORID DUCTAL EPITHELIAL HYPERPLASIA AND FIBROCYSTIC CHANGES WITH ASSOCIATED. MICROCALCIFICATIONS. L. PREVIOUS BIOPSY SITE CHANGES. M. ONE (1) INTRAMAMMARY IYYPH NODE, NEGATIVE FOR CARCINOMA (0/1). N. THE INVASIVE TUMOR CELLS ARE POSITIVE FOR ESTROGEN AND PROGESTERONE RECEPTORS. AND NEGATIVE FOR HER-2 (FISH NOT AMPLIFIED) AS PER PREVIOUS PATHOLOGY REPORT. PART 2: SENTINEL LYMPH NODE #1. RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA. (0/1). earcin oma, infiltrating dutal, NOS 8500/3. PART 3: SENTINEL LYMPH NODE #2, RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA (0/1). PART 4: SENTINEL LYMPH NODE #3. RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA (0/1). PART 5: RIGHT BREAST, 5 O' CLOCK, NEW MEDIAL MARGIN, EXCISION. A. NO CARCINOMA SEEN. B. BENIGN BREAST PARENCHYMA WITH FIBROCYSTIC CHANGES."



prompt = "As a professional pathologist, you will clean and reorganize the OCR-extracted TCGA pathology report into a single paragraph of plain text. The process will: Remove page numbers, headers/footers, and OCR artifacts. Correct common OCR errors. Merge fragmented text and maintain medical accuracy. Preserve all key pathological findings. Output as continuous plain text without markdown."

repot = "T. P.6/8. Anonymous No.: Gender: F. Race: White. CLINICAL HISTORY. Not provided. LMP: Hysterectomy. PRE-OP DIAGNOSIS: Right breast cancer. POST-OP DIAGNOSIS: Same. PROCEDURE: Right needle localized segmental mestectomy and right sentinel node biopsy. FINAL DIAGNOSIS. PART 1: RIGHT BREAST. 4 O' CLOCK NEEDLE LOCALIZED SEGMENTAL MASTECTOMY. A. TWO (2) FOCI OF INVASIVE DUCTALCARCINOMA NO SPECIAL TYPE. B. LARGER TUMOR NOTTINGHAM GRADE 2 (TUBULE FORMATION 3, NUCLEAR PLEOMORPHISM 2,. MITOTIC ACTIVITY 2; TOTAL SCORE: 7/9). C. SMALLER TUMOR NOTTINGHAM GRADE 2 (TUBULE FORMATION 3. NUCLEAR PLEOMORPHISM 2,. MITOTIC ACTIVITY 1; TOTAL SCORE: 6/9). THE LARGER TUMOR MEASURES 1.7 CM AND IS LOCATED AT 4 O' CLOCK. E. THE SMALLER TUMOR MEASURES 0.7 CM AND IS LOCATED AT 5 O' CLOCK. F. DUCTAL CARCINOMA IN SITU (DCIS), NUCLEAR GRADE 2. SOLID TYPE WITH MINIMAL NECROSIS. G. THE DCIS CONSTITUTES 5% OF THE TOTAL TUMOR MASS AND IS PRESENT AWAY FROM THE. INVASIVE COMPONENT. H. LYMPHOVASCULAR SPACE INVASION IS IDENTIFIED. I. INKED MARGINS ARE NEGATIVE FOR CARCINOMA. HOWEVER, INVASIVE CARCINOMA IS 1.0 MM TC. THE ANTERIOR MARGIN AND 1 5 MM TO THE POSTERIOR MARGIN (see comment). J. ATYPICAL DUCTAL HYPERPLASIA. K. FLORID DUCTAL EPITHELIAL HYPERPLASIA AND FIBROCYSTIC CHANGES WITH ASSOCIATED. MICROCALCIFICATIONS. L. PREVIOUS BIOPSY SITE CHANGES. M. ONE (1) INTRAMAMMARY IYYPH NODE, NEGATIVE FOR CARCINOMA (0/1). N. THE INVASIVE TUMOR CELLS ARE POSITIVE FOR ESTROGEN AND PROGESTERONE RECEPTORS. AND NEGATIVE FOR HER-2 (FISH NOT AMPLIFIED) AS PER PREVIOUS PATHOLOGY REPORT. PART 2: SENTINEL LYMPH NODE #1. RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA. (0/1). earcin oma, infiltrating dutal, NOS 8500/3. PART 3: SENTINEL LYMPH NODE #2, RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA (0/1). PART 4: SENTINEL LYMPH NODE #3. RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA (0/1). PART 5: RIGHT BREAST, 5 O' CLOCK, NEW MEDIAL MARGIN, EXCISION. A. NO CARCINOMA SEEN. B. BENIGN BREAST PARENCHYMA WITH FIBROCYSTIC CHANGES."
response = get_completion(prompt + " the pathology  repot:" + repot, 
                                           model=model_name
			                                     )
print('reasoning_content',response)


reasoning_content ChatCompletion(id='chatcmpl-433203c3043c4f1392d0815038208295', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Anonymous No.: Gender: F. Race: White. CLINICAL HISTORY: Not provided. LMP: Hysterectomy. PRE-OP DIAGNOSIS: Right breast cancer. POST-OP DIAGNOSIS: Same. PROCEDURE: Right needle localized segmental mastectomy and right sentinel node biopsy. FINAL DIAGNOSIS: PART 1: RIGHT BREAST. 4 O' CLOCK NEEDLE LOCALIZED SEGMENTAL MASTECTOMY. Two foci of invasive ductal carcinoma no special type were identified. The larger tumor has a Nottingham grade of 2 with tubule formation 3, nuclear pleomorphism 2, and mitotic activity 2, resulting in a total score of 7/9. This tumor measures 1.7 cm and is located at 4 o'clock. The smaller tumor also has a Nottingham grade of 2 with tubule formation 3, nuclear pleomorphism 2, and mitotic activity 1, resulting in a total score of 6/9. It measures 0.7 cm and is located at 5 o'clock. D

In [7]:
print('content',response.usage)

content CompletionUsage(completion_tokens=492, prompt_tokens=877, total_tokens=1369, completion_tokens_details=None, prompt_tokens_details=None)


# CUDA_VISIBLE_DEVICES=0,1 vllm serve /mnt/sdb/models/Qwen/Qwen2.5-VL/Qwen2.5-VL-32B-Instruct/ --tensor-parallel-size 2 --port 9999 --host 0.0.0.0 --gpu-memory-utilization 0.9 --max-model-len 32768 --served-model-name Qwen2.5-VL-32B-Instruct

In [11]:
from openai import OpenAI
import openai

openai.api_key = '1111111' # 这里随便填一个
openai.base_url = 'http://10.26.65.226:9999/v1'
model_name = 'Qwen2.5-VL-32B-Instruct'


def get_completion(prompt, model="Qwen3-32B-FP8"):
    client = OpenAI(api_key=openai.api_key,
                    base_url=openai.base_url
                    )
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=False
    )
    return response

prompt = "你是一个专业的病理医生,我现在有一个TCGA的病理报告,但是是用ocr从pdf中直接提取的, 因此可能存在问题,请帮我去除错误(例如可能存在页数被扫描进了文本或者报告的书面格式导致的问题),重新组织语言将其整理为一段话,要求是一段txt文本描述, 不能是markdown格式,报告内容如下:T. P.6/8. Anonymous No.: Gender: F. Race: White. CLINICAL HISTORY. Not provided. LMP: Hysterectomy. PRE-OP DIAGNOSIS: Right breast cancer. POST-OP DIAGNOSIS: Same. PROCEDURE: Right needle localized segmental mestectomy and right sentinel node biopsy. FINAL DIAGNOSIS. PART 1: RIGHT BREAST. 4 O' CLOCK NEEDLE LOCALIZED SEGMENTAL MASTECTOMY. A. TWO (2) FOCI OF INVASIVE DUCTALCARCINOMA NO SPECIAL TYPE. B. LARGER TUMOR NOTTINGHAM GRADE 2 (TUBULE FORMATION 3, NUCLEAR PLEOMORPHISM 2,. MITOTIC ACTIVITY 2; TOTAL SCORE: 7/9). C. SMALLER TUMOR NOTTINGHAM GRADE 2 (TUBULE FORMATION 3. NUCLEAR PLEOMORPHISM 2,. MITOTIC ACTIVITY 1; TOTAL SCORE: 6/9). THE LARGER TUMOR MEASURES 1.7 CM AND IS LOCATED AT 4 O' CLOCK. E. THE SMALLER TUMOR MEASURES 0.7 CM AND IS LOCATED AT 5 O' CLOCK. F. DUCTAL CARCINOMA IN SITU (DCIS), NUCLEAR GRADE 2. SOLID TYPE WITH MINIMAL NECROSIS. G. THE DCIS CONSTITUTES 5% OF THE TOTAL TUMOR MASS AND IS PRESENT AWAY FROM THE. INVASIVE COMPONENT. H. LYMPHOVASCULAR SPACE INVASION IS IDENTIFIED. I. INKED MARGINS ARE NEGATIVE FOR CARCINOMA. HOWEVER, INVASIVE CARCINOMA IS 1.0 MM TC. THE ANTERIOR MARGIN AND 1 5 MM TO THE POSTERIOR MARGIN (see comment). J. ATYPICAL DUCTAL HYPERPLASIA. K. FLORID DUCTAL EPITHELIAL HYPERPLASIA AND FIBROCYSTIC CHANGES WITH ASSOCIATED. MICROCALCIFICATIONS. L. PREVIOUS BIOPSY SITE CHANGES. M. ONE (1) INTRAMAMMARY IYYPH NODE, NEGATIVE FOR CARCINOMA (0/1). N. THE INVASIVE TUMOR CELLS ARE POSITIVE FOR ESTROGEN AND PROGESTERONE RECEPTORS. AND NEGATIVE FOR HER-2 (FISH NOT AMPLIFIED) AS PER PREVIOUS PATHOLOGY REPORT. PART 2: SENTINEL LYMPH NODE #1. RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA. (0/1). earcin oma, infiltrating dutal, NOS 8500/3. PART 3: SENTINEL LYMPH NODE #2, RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA (0/1). PART 4: SENTINEL LYMPH NODE #3. RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA (0/1). PART 5: RIGHT BREAST, 5 O' CLOCK, NEW MEDIAL MARGIN, EXCISION. A. NO CARCINOMA SEEN. B. BENIGN BREAST PARENCHYMA WITH FIBROCYSTIC CHANGES."



prompt = "As a professional pathologist, you will clean and reorganize the OCR-extracted TCGA pathology report into a single paragraph of plain text. The process will: Remove page numbers, headers/footers, and OCR artifacts. Correct common OCR errors. Merge fragmented text and maintain medical accuracy. Preserve all key pathological findings. Output as continuous plain text without markdown."

repot = "T. P.6/8. Anonymous No.: Gender: F. Race: White. CLINICAL HISTORY. Not provided. LMP: Hysterectomy. PRE-OP DIAGNOSIS: Right breast cancer. POST-OP DIAGNOSIS: Same. PROCEDURE: Right needle localized segmental mestectomy and right sentinel node biopsy. FINAL DIAGNOSIS. PART 1: RIGHT BREAST. 4 O' CLOCK NEEDLE LOCALIZED SEGMENTAL MASTECTOMY. A. TWO (2) FOCI OF INVASIVE DUCTALCARCINOMA NO SPECIAL TYPE. B. LARGER TUMOR NOTTINGHAM GRADE 2 (TUBULE FORMATION 3, NUCLEAR PLEOMORPHISM 2,. MITOTIC ACTIVITY 2; TOTAL SCORE: 7/9). C. SMALLER TUMOR NOTTINGHAM GRADE 2 (TUBULE FORMATION 3. NUCLEAR PLEOMORPHISM 2,. MITOTIC ACTIVITY 1; TOTAL SCORE: 6/9). THE LARGER TUMOR MEASURES 1.7 CM AND IS LOCATED AT 4 O' CLOCK. E. THE SMALLER TUMOR MEASURES 0.7 CM AND IS LOCATED AT 5 O' CLOCK. F. DUCTAL CARCINOMA IN SITU (DCIS), NUCLEAR GRADE 2. SOLID TYPE WITH MINIMAL NECROSIS. G. THE DCIS CONSTITUTES 5% OF THE TOTAL TUMOR MASS AND IS PRESENT AWAY FROM THE. INVASIVE COMPONENT. H. LYMPHOVASCULAR SPACE INVASION IS IDENTIFIED. I. INKED MARGINS ARE NEGATIVE FOR CARCINOMA. HOWEVER, INVASIVE CARCINOMA IS 1.0 MM TC. THE ANTERIOR MARGIN AND 1 5 MM TO THE POSTERIOR MARGIN (see comment). J. ATYPICAL DUCTAL HYPERPLASIA. K. FLORID DUCTAL EPITHELIAL HYPERPLASIA AND FIBROCYSTIC CHANGES WITH ASSOCIATED. MICROCALCIFICATIONS. L. PREVIOUS BIOPSY SITE CHANGES. M. ONE (1) INTRAMAMMARY IYYPH NODE, NEGATIVE FOR CARCINOMA (0/1). N. THE INVASIVE TUMOR CELLS ARE POSITIVE FOR ESTROGEN AND PROGESTERONE RECEPTORS. AND NEGATIVE FOR HER-2 (FISH NOT AMPLIFIED) AS PER PREVIOUS PATHOLOGY REPORT. PART 2: SENTINEL LYMPH NODE #1. RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA. (0/1). earcin oma, infiltrating dutal, NOS 8500/3. PART 3: SENTINEL LYMPH NODE #2, RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA (0/1). PART 4: SENTINEL LYMPH NODE #3. RIGHT AXILLA, BIOPSY. ONE (1) LYMPH NODE NEGATIVE FOR METASTATIC CARCINOMA (0/1). PART 5: RIGHT BREAST, 5 O' CLOCK, NEW MEDIAL MARGIN, EXCISION. A. NO CARCINOMA SEEN. B. BENIGN BREAST PARENCHYMA WITH FIBROCYSTIC CHANGES."
response = get_completion(prompt + " the pathology  repot:" + repot, 
                                           model=model_name
			                                     )
print('reasoning_content',response)


reasoning_content ChatCompletion(id='chatcmpl-c145ebbf79e945b89b112a56f97e1fd2', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="**Pathology Report Summary**\n\nThe patient is an anonymous female of white race with a history of hysterectomy. The pre-operative diagnosis was right breast cancer, confirmed post-operatively. The procedure performed included a right needle-localized segmental mastectomy at the 4 o'clock position and a right sentinel lymph node biopsy.\n\n**Final Diagnosis - Part 1: Right Breast**\n- **Needle-Localized Segmental Mastectomy (4 o'clock):**\n  - Two foci of invasive ductal carcinoma, no special type:\n    - Larger tumor: Nottingham Grade 2 (tubule formation 3, nuclear pleomorphism 2, mitotic activity 2; total score: 7/9). Measures 1.7 cm.\n    - Smaller tumor: Nottingham Grade 2 (tubule formation 3, nuclear pleomorphism 2, mitotic activity 1; total score: 6/9). Measures 0.7 cm and located at 5 o'clock.\n  - D

In [10]:
print('content',response.usage)

content CompletionUsage(completion_tokens=535, prompt_tokens=877, total_tokens=1412, completion_tokens_details=None, prompt_tokens_details=None)


# Docker

docker run -d --gpus all \
--name zhangchen_ubuntu \
-u 1152:968 \
-v /etc/passwd:/etc/passwd:ro \
-v /etc/group:/etc/group:ro \
-p 7000:7000 \
-p 7001:7001 \
-p 7002:7002 \
-p 7003:7003 \
-p 7004:7004 \
-p 7005:7005 \
-p 7006:7006 \
-p 7007:7007 \
-p 7008:7008 \
-p 7009:7009 \
-p 7010:7010 \
-v /mnt/mydisk/zhangchen/env/docker_ubuntu:/home/zhangchen \
-v /mnt/raid5_lyx/zhangchen/env/docker_ubuntu:/mnt/sda \
-v /mnt/data/zhangchen/env/docker_ubuntu:/mnt/sdb \
-w /home/zhangchen \
f95a9dd823d3 \
tail -f /dev/null

In [ ]:
docker run -d --gpus all \
--name zhangchen_ubuntu \
-u 1152:968 \
-v /etc/passwd:/etc/passwd:ro \
-v /etc/group:/etc/group:ro \
-p 7000:7000 \
-p 7001:7001 \
-p 7002:7002 \
-p 7003:7003 \
-p 7004:7004 \
-p 7005:7005 \
-p 7006:7006 \
-p 7007:7007 \
-p 7008:7008 \
-p 7009:7009 \
-p 7010:7010 \
-v /mnt/mydisk/zhangchen/env/docker_ubuntu:/home/zhangchen \
-v /mnt/raid5_lyx/zhangchen/env/docker_ubuntu:/mnt/sda \
-v /mnt/data/zhangchen/env/docker_ubuntu:/mnt/sdb \
-w /home/zhangchen \
f95a9dd823d3 \
tail -f /dev/null